In [ ]:
import pandas as pd
from pathlib import Path
import re
import os


def load_file_based_labels(
    years: list[str],
    suffix: str,
    prefix: str = "_llm-labeled",
    path: str = "llm_labeled",
):
    df = pd.DataFrame()

    # create the folder and filename regex
    folder = Path(path)
    if suffix != "":
        suffix = "_" + suffix
    filename_regex = rf"{prefix}-{'-'.join(years)}.*{suffix}\.csv"
    print(filename_regex)

    # loop through all the meta files
    for file in os.listdir(folder):
        filename = os.fsdecode(file)
        # if the filename matches the regex, load it
        if re.match(filename_regex, filename):
            # load the dataframe part using pandas
            df_part = pd.read_csv(folder / file, header=0, index_col=0)
            # concatenate with full dataframe
            if len(df) != 0:
                df = pd.concat([df, df_part])
            else:
                df = df_part

    return df

In [ ]:
from extract_for_llm import extract_for_llm

# get the full dataframe
df_speech = extract_for_llm(nr_lines=-1)

# load in the labeled CSVs and unite it into one
df_llm = load_file_based_labels(
    ["2017"], "gemini_flash", path="llm_labeled\\llm_labeled_no_id"
)

_llm-labeled-2017.*_gemini_flash\.csv
lf_gemini_flash_v1_annotator 0
discovery_pro-2017.*\.csv
lf_discovery_pro 2180
discovery_contra-2017.*\.csv
lf_discovery_contra 920
discovery_neutral-2017.*\.csv
lf_discovery_neutral 860
Extracting speeches...


  0%|          | 0/6077 [00:00<?, ?it/s]

Filtering 123267 speeches based on keywords...
Filtering 116741 speeches based on full text dimensions...
Filtering 20603 speeches based on sentence dimensions...
Transforming 15126 speeches to -1...
_llm-labeled-2017.*_gemini_flash\.csv


In [4]:
df_llm.head()

,label,confidence,reason,Text_ID,ID
0,NEUTRAL,1.0,The sentence is a formal address to the Speake...,ParlaMint-HU_2017-02-20,u2017-02-20-39
1,NEUTRAL,1.0,"The sentence is a purely factual, procedural a...",ParlaMint-HU_2017-02-20,u2017-02-20-39
2,NEUTRAL,1.0,This sentence is a factual administrative anno...,ParlaMint-HU_2017-02-20,u2017-02-20-39
3,NEUTRAL,0.9,The sentence critiques the Prime Minister's of...,ParlaMint-HU_2017-02-20,u2017-02-20-43
4,NEUTRAL,1.0,The sentence factually reports a court request...,ParlaMint-HU_2017-02-20,u2017-02-20-47


In [19]:
# make the common ID
df_speech["_ID"] = df_speech["ID"].map(lambda id: "-".join(id.split("-")[:-1]))
df_speech["Common_ID"] = df_speech.groupby("_ID").cumcount()

df_llm["Common_ID"] = df_llm.groupby("ID").cumcount()

In [20]:
df_speech.head()

,full_speech,Text_ID,ID,sentence,Common_ID,_ID
753,"„Kövér László úr, az Országgyűlés elnöke részé...",ParlaMint-HU_2017-02-20,u2017-02-20-39-1,"„Kövér László úr, az Országgyűlés elnöke részére.",0,u2017-02-20-39
755,"„Kövér László úr, az Országgyűlés elnöke részé...",ParlaMint-HU_2017-02-20,u2017-02-20-39-3,"Tájékoztatom az Országgyűlést, hogy az Alaptör...",1,u2017-02-20-39
756,"„Kövér László úr, az Országgyűlés elnöke részé...",ParlaMint-HU_2017-02-20,u2017-02-20-39-4,Egyúttal az Alaptörvény 9. cikk (4) bekezdés j...,2,u2017-02-20-39
763,"Köszönöm, elnök úr. Azt indítványozom, hogy el...",ParlaMint-HU_2017-02-20,u2017-02-20-43-2,"Azt indítványozom, hogy elnök úr hívja össze a...",0,u2017-02-20-43
774,Tisztelt Elnök Úr! Köszönöm a szót. Tisztelt H...,ParlaMint-HU_2017-02-20,u2017-02-20-47-5,A Jászberényi Járásbíróság az előtte 13.Bpk.92...,0,u2017-02-20-47


In [21]:
df_llm.head()

,label,confidence,reason,Text_ID,ID,Common_ID
0,NEUTRAL,1.0,The sentence is a formal address to the Speake...,ParlaMint-HU_2017-02-20,u2017-02-20-39,0
1,NEUTRAL,1.0,"The sentence is a purely factual, procedural a...",ParlaMint-HU_2017-02-20,u2017-02-20-39,1
2,NEUTRAL,1.0,This sentence is a factual administrative anno...,ParlaMint-HU_2017-02-20,u2017-02-20-39,2
3,NEUTRAL,0.9,The sentence critiques the Prime Minister's of...,ParlaMint-HU_2017-02-20,u2017-02-20-43,0
4,NEUTRAL,1.0,The sentence factually reports a court request...,ParlaMint-HU_2017-02-20,u2017-02-20-47,0


In [22]:
# cross-reference
ids = []
for _, row in df_llm.iterrows():
    same_id = df_speech[
        (df_speech["_ID"] == row["ID"]) & (df_speech["Common_ID"] == row["Common_ID"])
    ]
    if len(same_id) > 0:
        ids.append(same_id.iloc[0]["ID"])

ids[:5]

['u2017-02-20-39-1',
 'u2017-02-20-39-3',
 'u2017-02-20-39-4',
 'u2017-02-20-43-2',
 'u2017-02-20-47-5']

In [23]:
df_llm["ID"] = ids

In [25]:
del df_llm["Common_ID"]

In [26]:
from utilities import create_file_name

df_llm.to_csv(
    create_file_name(
        ".\\llm_labeled\\_llm_labeled",
        ["2017"],
        f"{0}-{2700}_gemini_flash",
        "csv",
    )
)

## Setup

In [ ]:
from collections import Counter
from utilities import load_docs
import spacy

In [4]:
nlp = spacy.load(
    "hu_core_news_md",
    disable=[
        # "tok2vec",
        "senter",
        # "tagger",
        # "morphologizer",
        # "lookup_lemmatizer",
        # "trainable_lemmatizer",
        # "parser",
        "ner",
    ],
)

In [5]:
def get_lemma_freq(docs):
    counter = Counter()
    for doc in docs:
        for t in doc:
            if not t.is_stop and t.is_alpha:
                counter[t.lemma_.lower()] += 1
    return counter

In [13]:
def cooccurrence(docs, seed_words, window=5, min_length=3):
    cooc = Counter()

    for doc in docs:
        lemmas = [t.lemma_.lower() for t in doc]
        for i, tok in enumerate(lemmas):
            if tok in seed_words:
                for j in range(max(0, i - window), min(len(lemmas), i + window)):
                    if j != i and len(lemmas[j]) > min_length:
                        cooc[lemmas[j]] += 1
    return cooc

## Data Loading

In [7]:
docs = load_docs(nlp, ["2017"], suffix="2017-02-22")

0it [00:00, ?it/s]

## Data Analysis

In [16]:
cooc = cooccurrence(docs, ["hatalom"])
print(cooc.most_common(30))

[('olyan', 3), ('végrehajtó', 3), ('hogy', 3), ('kerül', 2), ('közigazgatás', 2), ('kontroll', 2), ('mond', 2), ('amely', 1), ('pénz', 1), ('minden', 1), ('képes', 1), ('2018', 1), ('jutó', 1), ('jobbik-kormány', 1), ('ahol', 1), ('csak', 1), ('kicsi', 1), ('közel', 1), ('férkőz', 1), ('egyből', 1), ('kormányzat', 1), ('többség', 1), ('bíróság', 1), ('alatt', 1), ('bírósági', 1), ('gyakorol', 1), ('tevékenység', 1), ('felett', 1), ('közigazgatási', 1), ('rogán', 1)]
